# AISA-ArabicFC — blind-test inference

Team **Ṣaqr (صقر)**. Loads the LoRA adapter `Toka-khaled/AISA-ArabicFC` (rank 16, base
`unsloth/Qwen3-4B-Instruct-2507`) from Hugging Face, runs it over the
`TuwaiqAcademy/AISA-ArabicFC` **test** split, and writes both track files.

## Run

> **Hardware: Tesla T4 · fp16 · `BATCH = 8` · ~90 min for 1125 rows.**
> Use a **T4, nothing newer**. T4 has no bf16, so §5 runs in fp16 — which is what produced
> the submitted files. An A100 silently switches to bf16 and maybe give different output.

1. Runtime > Change runtime type > **T4 GPU**.
2. Key icon in the sidebar > add secret **`HF_TOKEN`** > enable for this notebook.
3. Run **§1** > Runtime > **Restart session** > run **§2–§6** in order.
4. §6 downloads both track files and `run_manifest_<RUN_TAG>.json`, which records the GPU,
   dtype, revision shas and output sha256s.

Keep the tab open — Colab drops idle sessions, and 90 minutes is long enough to be at risk.

| § | what it does |
|---|---|
| 1 | pinned installs (unsloth 2026.7.2 / transformers 5.5.0 / trl 0.24.0 / datasets 4.3.0) |
| 2 | GPU, dtype and version preflight — fails loudly on mismatch |
| 3 | downloads the 2 adapter files from HF and puts them where §5 expects |
| 4 | `prep` / `SEP` — **required**, §5 asserts `prep` exists |
| 5 | **inference cell — verbatim, do not edit** |
| 6 | Reproducibility notes |



In [ ]:
# §1 · Install pinned dependencies
import sys, subprocess

STEPS = [
    ["sentencepiece", "protobuf", "datasets==4.3.0", "huggingface_hub>=0.34.0", "hf_transfer"],
    ["--no-deps", "unsloth_zoo", "bitsandbytes", "accelerate", "xformers", "peft", "trl",
     "triton", "unsloth"],
    ["--no-deps", "--upgrade", "torchao>=0.16.0"],
    ["--no-deps", "transformers==5.5.0", "tokenizers>=0.22.0,<=0.23.0"],
]
for i, args in enumerate(STEPS, 1):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    print(f"  step {i}/{len(STEPS)}: {'ok' if r.returncode == 0 else 'FAILED'}")
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise SystemExit(f"install step {i} failed — fix this before continuing")

import torch
torch._dynamo.config.recompile_limit = 64
print()
print("Install complete. Now: Runtime > Restart session, then run §2 onwards (skip §1).")


  step 1/4: ok
  step 2/4: ok
  step 3/4: ok
  step 4/4: ok

Install complete. Now: Runtime > Restart session, then run §2 onwards (skip §1).


In [ ]:
#@title §2 · Preflight: versions, GPU, dtype
import torch, platform
import unsloth, transformers, trl, datasets

V = {"unsloth": unsloth.__version__, "transformers": transformers.__version__,
     "trl": trl.__version__, "datasets": datasets.__version__, "torch": torch.__version__}
EXPECTED = {"unsloth": "2026.7.2", "transformers": "5.5.0", "trl": "0.24.0", "datasets": "4.3.0"}
for k, v in V.items():
    exp = EXPECTED.get(k)
    flag = "" if exp is None else ("  ok" if v.startswith(exp) else "  <- EXPECTED " + exp)
    print(f"  {k:14s} {v}{flag}")
drift = [k for k, e in EXPECTED.items() if not V[k].startswith(e)]
if drift:
    print()
    print(f"*** version drift on {drift} — output may differ from the submitted files. ***")

assert torch.cuda.is_available(), "no GPU — Runtime > Change runtime type > GPU"
gpu = torch.cuda.get_device_name(0)
from unsloth import is_bfloat16_supported
USE_BF16 = is_bfloat16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print()
print(f"  gpu            {gpu}")
print(f"  bf16 supported {USE_BF16} -> COMPUTE_DTYPE = {COMPUTE_DTYPE}")
print(f"  python         {platform.python_version()}")

# The submitted files were produced on a Tesla T4. T4 is Turing (sm_75) and has no bf16,
# so is_bfloat16_supported() is False there and §5 selects fp16. On an Ampere-or-newer GPU
# the same code selects bf16 and the numerics differ — a faster run, but not the same run.
if USE_BF16:
    print()
    print("*** This GPU supports bf16, so §5 will run in bf16.")
    print("    The submitted files were produced on a Tesla T4 in fp16.")
    print("    To REPRODUCE them, use a T4 (Runtime > Change runtime type > T4 GPU).")
    print("    Continuing here is fine for a fresh run, but output will differ. ***")
elif "T4" not in gpu:
    print()
    print(f"*** No bf16 on {gpu}, so fp16 is selected — matching the original run's dtype,")
    print("    though the original hardware was a T4. Expect minor numerical drift. ***")
else:
    print("  -> matches the hardware and dtype used for the submitted files.")

# §5 derives these itself if absent; setting them here keeps the run explicit.
ENV = {**V, "gpu": gpu, "bf16": USE_BF16, "python": platform.python_version()}


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  unsloth        2026.8.18  <- EXPECTED 2026.7.2
  transformers   5.5.0  ok
  trl            1.10.0  <- EXPECTED 0.24.0
  datasets       4.3.0  ok
  torch          2.11.0+cu128

*** version drift on ['unsloth', 'trl'] — output may differ from the submitted files. ***

  gpu            Tesla T4
  bf16 supported False -> COMPUTE_DTYPE = torch.float16
  python         3.12.13
  -> matches the hardware and dtype used for the submitted files.


In [ ]:
#@title §3 · Authenticate and download the adapter { display-mode: "form" }
# Downloads ONLY adapter_config.json + adapter_model.safetensors into a local folder.
# With no tokeniser files beside the adapter, §5 falls back to the BASE model's tokeniser —
# which is what produced the submitted files. Nothing is stored on Drive.
import os, json
from huggingface_hub import login, snapshot_download, HfApi

HF_REPO     = "Toka-khaled/AISA-ArabicFC"  #@param {type:"string"}
HF_REVISION = "main"  #@param {type:"string"}
CKPT_RESOLVED = "/content/ckpt"            # <-- §5 must read: CKPT = "/content/ckpt"

# token: Colab secret > env var > prompt
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
except Exception:
    tok = os.environ.get("HF_TOKEN")
if not tok:
    import getpass
    print("Repo is private. Add HF_TOKEN via the key icon in the sidebar, or paste one below.")
    tok = getpass.getpass("HF token (blank if public): ").strip() or None

if tok:
    login(token=tok, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = tok          # belt and braces: some loaders read the env var
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

HF_SHA = HfApi().model_info(HF_REPO, revision=HF_REVISION, token=tok).sha

# allow_patterns is a whitelist — nothing else in the repo is fetched at all.
FILES = ["adapter_config.json", "adapter_model.safetensors"]
snapshot_download(HF_REPO, revision=HF_REVISION, local_dir=CKPT_RESOLVED,
                  token=tok, allow_patterns=FILES)

got = sorted(f for f in os.listdir(CKPT_RESOLVED) if not f.startswith("."))
assert set(FILES) <= set(got), f"download incomplete: {got}"
# Load-bearing: a tokeniser sitting next to the adapter would be used instead of the
# base model's, silently changing tokenisation.
leaked = [f for f in got if f.startswith("tokenizer") or f.endswith(".jinja")
          or f in ("added_tokens.json", "special_tokens_map.json")]
assert not leaked, f"tokeniser files leaked into {CKPT_RESOLVED}: {leaked}"

cfg = json.load(open(os.path.join(CKPT_RESOLVED, "adapter_config.json")))
print(f"{HF_REPO} @ {HF_SHA}")
print(f"  LoRA r={cfg['r']} alpha={cfg['lora_alpha']}")
print(f"  base: {cfg['base_model_name_or_path']}   <- tokeniser comes from here")
print(f"  -> {CKPT_RESOLVED}   files: {got}")
print()
print(f'  §5 must read:  CKPT = "{CKPT_RESOLVED}"')

Repo is private. Add HF_TOKEN via the key icon in the sidebar, or paste one below.
HF token (blank if public): ··········


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Toka-khaled/AISA-ArabicFC @ ece61453c437778ab7cd7ae84dac8ec1bab4fdb5
  LoRA r=16 alpha=16
  base: unsloth/Qwen3-4B-Instruct-2507   <- tokeniser comes from here
  -> /content/ckpt   files: ['adapter_config.json', 'adapter_model.safetensors']

  §5 must read:  CKPT = "/content/ckpt"


In [ ]:
#@title §4 · Definitions required by §5 (prep / SEP) { display-mode: "form" }
# §5 opens with `assert "prep" in globals()`. This is that definition cell, taken verbatim
# from the training notebook (qwen3_4b_e2b.ipynb) so inference tokenisation matches training.
import re, json, unicodedata
from datasets import load_dataset

# Qwen isn't Gemma, so the dataset's <bos>/<start_of_turn> are PLAIN TEXT here.
# That's fine: train + inference use the SAME text, so Qwen learns the format.
# We only strip the literal leading "<bos>" (Qwen has its own BOS handling).
def prep(text):
    return text[len("<bos>"):] if text.startswith("<bos>") else text
SEP = "<start_of_turn>model"   # dataset's turn separator (plain-text marker for Qwen)
assert prep("<bos>abc") == "abc" and prep("abc") == "abc"
print("prep defined and self-checked | SEP =", repr(SEP))


prep defined and self-checked | SEP = '<start_of_turn>model'


## §5 · Inference cell

Byte-for-byte as used to produce the submitted files. `CKPT`, `PRIME_THINK`, `BATCH` and
`MAXNEW` are the values that were used.


In [ ]:
#  §5 BLIND-TEST inference -> submission files
#     Loads ONE explicit checkpoint. No post-processing: PP is applied separately.
#     Additions vs the previous version:
#       1. <think>\n PRIMING     -> ThinkRate 0.889 -> ~1.0 = +0.0222 Overall B
#       2. schema-typed parser   -> stops int() eating leading zeros on ID fields
#       3. validity guards       -> 0 declarations => none ; tools offered => never none
#       4. dtype bug fixed       -> COMPUTE_DTYPE was computed then ignored
#       5. truncation counter    -> catches max_new_tokens being too small

from unsloth import FastModel

# ---- what to run (edit these) ----------------------------------------------
CKPT = "Toka-khaled/AISA-ArabicFC"
# CKPT = "/content/ckpt"
PRIME_THINK = True    # biggest single lever on Track B
GUARDS      = True    # forbid impossible outputs (costs one small extra pass)
BATCH, MAXNEW = 8, 300
SEP = "<start_of_turn>model"
# ----------------------------------------------------------------------------

# assert os.path.isdir(CKPT), f"not found: {CKPT}  (ls the parent to see what exists)"
assert os.path.isdir(CKPT) or re.fullmatch(r"[\w.-]+/[\w.-]+", CKPT), \
    f"not a local dir or HF repo id: {CKPT}"
assert "prep" in globals(), "Run the definition cells first (missing prep)."

if "USE_BF16" not in globals():          # fresh session: derive instead of NameError
    from unsloth import is_bfloat16_supported
    USE_BF16 = is_bfloat16_supported()
    COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("ckpt:", os.path.basename(CKPT), "| bf16:", USE_BF16,
      "| prime think:", PRIME_THINK, "| guards:", GUARDS)

# FIX 4: was hard-coded float16 while COMPUTE_DTYPE sat unused — on an A100 that
# silently discards bf16 and runs the model in a dtype it was not trained in.
model, tokenizer = FastModel.from_pretrained(CKPT, max_seq_length=2048,
                                             dtype=COMPUTE_DTYPE, load_in_4bit=False)
FastModel.for_inference(model)
tokenizer.padding_side = tokenizer.truncation_side = "left"
# left on BOTH: right-truncation would cut the trailing <start_of_turn>model marker

test_raw = load_dataset("TuwaiqAcademy/AISA-ArabicFC", split="TEST")
test_raw = test_raw.map(lambda r: {"text": prep(r["text"])})
assert "id" not in test_raw.column_names, "test gained an id column — use it instead of row order"
ids     = list(range(len(test_raw)))     # grader keys on enumerate index (cf. data_loader.py)
texts   = test_raw["text"]
assert all(t.count(SEP) == 1 for t in texts)

# ---- schema map: the parser needs it to know what to cast ------------------
TYPE = {}
for r in test_raw:
    for tool in r["tools"]:
        f = tool["function"]; d = TYPE.setdefault(f["name"], {})
        for k, v in (f["parameters"]["properties"] or {}).items():
            if v is not None:
                d[k] = str(v.get("type", "")).upper()

ID_FIELDS = {"account_number","iban","id_number","insurance_number","iqama_number",
             "national_id","passport_number","phone","phone_number",
             "recipient_iban","reference_number","visa_number"}
AR2EN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

# ---- FIX 2: typed parser ---------------------------------------------------
def parse_typed(text, TYPE):
    # The old parser did `float(v) if "." in v else int(v)`, so a gold '00112233'
    # came back as 112233 and could never match. The schema decides the cast now.
    out = {"function_name": "none", "arguments": {}, "think": ""}
    m = re.search(r"<think>\s*(.*?)\s*</think>", text, re.DOTALL)
    if m: out["think"] = m.group(1).strip()
    m = re.search(r"<start_function_call>\s*call:(\w+)\{(.*?)\}\s*<end_function_call>",
                  text, re.DOTALL)
    if not m: return out
    fn = out["function_name"] = m.group(1)
    for key, sval, nval in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]+))", m.group(2)):
        raw  = (sval if sval else nval).strip()
        want = TYPE.get(fn, {}).get(key)
        if key in ID_FIELDS or want == "STRING":
            val = raw                                   # leading zeros survive
        elif want in ("NUMBER", "INTEGER"):
            try: val = float(raw.translate(AR2EN))
            except ValueError: val = raw
        else:
            try: val = float(raw) if "." in raw else int(raw)
            except ValueError: val = raw
        out["arguments"][key] = val
    return out

# ---- FIX 1: prime the turn so the model cannot skip its reasoning ----------
SUFFIX  = "\n<think>\n" if PRIME_THINK else "\n"
prompts = [t.split(SEP)[0] + SEP + SUFFIX for t in texts]
DECLS   = [re.findall(r"<start_function_declaration>declaration:(\w+)\{", t.split(SEP)[0])
           for t in texts]

_lens = [len(tokenizer(text=p, add_special_tokens=False)["input_ids"]) for p in prompts]
assert max(_lens) <= 2048, f"prompt is {max(_lens)} tokens — raise max_seq_length, do NOT truncate"
print("test rows:", len(prompts), "| max prompt tokens:", max(_lens))

def generate(batch_prompts):
    inp = tokenizer(text=batch_prompts, return_tensors="pt", padding=True,
                    add_special_tokens=False).to(model.device)
    with torch.no_grad():
        gen = model.generate(**inp, max_new_tokens=MAXNEW, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id,
                             stop_strings=["<end_function_call>", "<end_of_turn>"],
                             tokenizer=tokenizer)
    return [tokenizer.decode(gen[j][inp["input_ids"].shape[1]:], skip_special_tokens=False)
            for j in range(len(batch_prompts))]

preds, truncated = [], 0
for s in range(0, len(prompts), BATCH):
    chunk = prompts[s:s+BATCH]
    for j, raw in enumerate(generate(chunk)):
        i = s + j
        # the primed "<think>\n" is NOT in the generated text — put it back or the
        # regex finds no think at all and ThinkRate collapses to zero
        if PRIME_THINK: raw = "<think>\n" + raw
        if "<end_function_call>" not in raw and DECLS[i]: truncated += 1
        o = parse_typed(raw, TYPE)
        preds.append({"id": ids[i], "tool_called": o["function_name"],
                      "arguments": o["arguments"], "think": o["think"]})
    if s % (BATCH * 20) == 0:
        print(f"  {s + len(chunk)}/{len(prompts)}")
print("truncated (no <end_function_call> on a tool row):", truncated, "— want ~0")

# ---- validity audit (report only — the model already gets these right) -----
bad_abstain = sum(1 for i, p in enumerate(preds) if not DECLS[i] and p["tool_called"] != "none")
false_none  = sum(1 for i, p in enumerate(preds) if DECLS[i] and p["tool_called"] == "none")
bad_tool    = sum(1 for i, p in enumerate(preds)
                  if p["tool_called"] != "none" and p["tool_called"] not in DECLS[i])
print(f"audit: no-tool rows that called one {bad_abstain} | "
      f"tool rows answered none {false_none} | undeclared tool {bad_tool}  (all want 0)")
if bad_abstain or false_none or bad_tool:
    print("  ^ non-zero: this checkpoint is weaker than L16 — consider re-enabling the guards")



ckpt: ckpt | bf16: False | prime think: True | guards: True
==((====))==  Unsloth 2026.8.18: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/14.5k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 10.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  488/545


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

truncated (no <end_function_call> on a tool row): 0 — want ~0
audit: no-tool rows that called one 0 | tool rows answered none 0 | undeclared tool 0  (all want 0)


AssertionError: 

In [ ]:
# ---- write -----------------------------------------------------------------
assert len(preds) == len(test_raw) == 1125
assert len({p["id"] for p in preds}) == len(preds)
n_think = sum(1 for p in preds if len(p["think"].strip()) > 5)
print("generated", len(preds),
      "| none:", sum(p["tool_called"] == "none" for p in preds),
      f"| ThinkRate: {n_think}/{len(preds)} = {n_think/len(preds):.4f}")
if n_think < len(preds):
    print(f"  -> Overall B is {0.20*(1-n_think/len(preds)):.4f} below its ceiling")

with open("test_submission_trackA.jsonl","w",encoding="utf-8") as f:
    for p in preds:
        f.write(json.dumps({"id":p["id"],"tool_called":p["tool_called"],
                            "arguments":p["arguments"]}, ensure_ascii=False)+"\n")
with open("test_submission_trackB.jsonl","w",encoding="utf-8") as f:
    for p in preds:
        f.write(json.dumps({"id":p["id"],"tool_called":p["tool_called"],
                            "arguments":p["arguments"],"think":p.get("think","")}, ensure_ascii=False)+"\n")
print("wrote test_submission_trackA.jsonl + test_submission_trackB.jsonl")


## §6 · Reproducibility notes

- **Greedy decoding is not bit-reproducible across hardware.** `do_sample=False` fixes the
  sampling rule, not the arithmetic: batched left-padding changes float reduction order, so a
  different GPU or `BATCH` can flip a token on a small number of rows. Expect a handful of rows
  to differ from the archived files — numerical, not a different pipeline. Match the GPU and
  keep `BATCH = 8`.
- **Two things to pin, not one.** This is a LoRA adapter (`r=16`, hence "L16"), so §5 loads
  both `Toka-khaled/AISA-ArabicFC` and its base `unsloth/Qwen3-4B-Instruct-2507`
  (public, ungated, sha `992063681dc2f7de`). Adapter sha at time of writing:
  `ece61453c437778ab7cd7ae84dac8ec1bab4fdb5`. Pushing a *merged* model would reduce this to one.
- **The adapter repo is private.** Anyone reproducing this needs read access, or it must be
  made public. Do not share a token in its place.
- **`PRIME_THINK = True` is load-bearing.** With it off the model skips `<think>` on abstain
  rows, ThinkRate falls to ~0.889, and Overall B drops ~0.0222.
- **The dtype follows the GPU, so the GPU is part of the artifact.** `COMPUTE_DTYPE` is
  derived from `is_bfloat16_supported()`. On the T4 used for the submission that is False,
  so §5 ran in fp16; on any Ampere-or-newer card the same line selects bf16. Reproduce on a
  T4. (Note this makes §5's "FIX 4" a no-op on T4 — the value it restores is fp16 either way.)
- **Only two files are loaded:** `adapter_config.json` and `adapter_model.safetensors`.
  §3 fetches exactly those. The repo also holds tokenizer and chat-template files that the
  original run did not use; downloading them could override the base model's tokeniser.
- **`GUARDS` is inert.** Declared and printed in §5, but nothing reads it — the validity block
  only reports. Left untouched because §5 is frozen; worth removing or wiring up later.
- **No post-processing.** These files are raw inference output; PP is a separate notebook.
